# 03 · G3 后的正式实验

默认 manifest-only。32 个核心作业仍需要真实 G3；新增 2026 强基线和消融尚不能由核心矩阵自动替代。

GitHub / VS Code 工作流先运行 `00_Colab_GitHub_Workspace.ipynb`；详见 [操作说明](../docs/COLAB_VSCODE.md)。已有 ZIP 入口仍可用。

## 路径与环境配置

GitHub / VS Code 首先运行 `00_Colab_GitHub_Workspace.ipynb`；本 notebook 会识别 `/content/CellQuant` 并读取运行时路径配置。
已有 ZIP 仍可设置 `KIT_ARCHIVE` 和 `EXTRACT_ARCHIVE=True` 解压，或直接指定 `PROJECT_ROOT`。
默认 smoke / manifest-only，不下载模型、不启动全量 sweep。基础依赖缺失时才设置 `INSTALL_BASE_DEPENDENCIES=True`，不替换宿主 torch。
真实模型授权通过环境变量或隐藏输入提供，不能写入 notebook 输出或 Git；阶段结束用 workspace backup 保存到 Drive。


In [ ]:
from pathlib import Path
import os, sys, json

# Colab: Git clone 到 /content/CellQuant；也兼容完整 ZIP 解压目录。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or ("/content/CellQuant" if Path("/content/CellQuant/pyproject.toml").is_file() else None)
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = Path("/content").is_dir()  # Colab 默认挂载，本地/AutoDL 默认跳过
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

## 挂载 Google Drive

运行下方单元，按 Google 弹窗完成登录授权，然后继续运行后续单元。
此命令同时用于浏览器 Colab 和 VS Code 的 Colab 内核；已挂载时会复用，非 Colab 环境默认跳过。
`CellQuant` 项目文件夹会在后续初始化中自动创建。


In [ ]:
# 在 Colab（浏览器或 VS Code 内核）中直接挂载 Google Drive。
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive 挂载尚未完成，请完成授权后重跑本单元。")
    print("Google Drive 已挂载：/content/drive/MyDrive")
else:
    print("非 Colab 环境或 MOUNT_DRIVE=False：跳过云盘挂载。")


In [ ]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache/huggingface"))
os.environ.setdefault("PYTHONUTF8", "1")

# Separate notebook kernels reuse the same local runtime path configuration.
workspace_record = PROJECT_ROOT / ".cache/colab_workspace.json"
if workspace_record.is_file():
    from cellquant.workspace import configure_workspace
    saved = json.loads(workspace_record.read_text(encoding="utf-8"))
    configure_workspace(PROJECT_ROOT, saved["environment"]["CELLQUANT_DRIVE_ROOT"])


先用 `scripts/prepare_matrix.py` 准备统一协议；G2 后在新目录准备冻结的正式 tokens，再生成绑定当前源码/环境/数据的 G3。整个流程见 `docs/RUNBOOK.md`。

In [ ]:
from cellquant.runtime import read_json
from cellquant.runner import make_job_manifest, check_execution_readiness, execute_manifest
OUT = PATHS["runs"] / "03_full"
OUT.mkdir(parents=True, exist_ok=True)
MATRIX_PATH = Path(os.environ.get("CELLQUANT_FULL_MATRIX", PROJECT_ROOT / "runs/prepared/full/matrix.json"))
if not MATRIX_PATH.is_file():
    MATRIX_PATH = PROJECT_ROOT / "configs/full_experiments.json"
cfg = read_json(MATRIX_PATH)
GATE_PATH = Path(os.environ.get("CELLQUANT_G3_REPORT", PROJECT_ROOT / cfg["gate_report_path"]))
GATE_EVIDENCE_PATH = Path(os.environ.get("CELLQUANT_G3_EVIDENCE", PROJECT_ROOT / cfg["gate_evidence_path"]))
plan = make_job_manifest(cfg["jobs"], project_root=PROJECT_ROOT, gpu_hour_budget=cfg["gpu_hour_budget"],
    required_gate="G3", execution_provenance_path=cfg["execution_provenance_path"])
atomic_json(OUT / "plan.json", plan)
print(check_execution_readiness(plan, gate_path=GATE_PATH, gate_evidence_path=GATE_EVIDENCE_PATH))


In [ ]:
RUN_FULL = False
execution = execute_manifest(plan, project_root=PROJECT_ROOT, gate_path=GATE_PATH,
    gate_evidence_path=GATE_EVIDENCE_PATH, state_dir=OUT / "state", allow_execution=RUN_FULL)
print(execution)


长作业通过可中断 subprocess、日志及心跳执行。KeyboardInterrupt 终止子进程；完整作业可按 hash 跳过。事件求解器、旋转训练和 GPTQ 有各自恢复检查；grid/STE 目前仅支持已完成方法恢复，不能宣称其中途状态已恢复。结果是反量化浮点模拟，不报告 packed INT4 加速或持久 KV 内存节省。